# **SPRSound** Dataset Preparation for Advanced Models
### Preparing SPRSound data with proper labels for:
### Model 1: Supervised Contrastive Learning ResNet
### Model 2: HeAR Foundation Model + Classifier


## 1. Set up Imports

In [14]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

# Add project root
sys.path.append('..')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')

## 2. Load SPRSound features

In [15]:
sprsound_features = pd.read_csv("../../sound_data/features/sprsound_features.csv")

print("="*60)
print("FEATURES LOADED")
print("="*60)

print(f"SPRSound features: {sprsound_features.shape}")
print(f"Columns: {list(sprsound_features.columns)[:10]}...")

sprsound_features.head()

FEATURES LOADED
SPRSound features: (6567, 35)
Columns: ['mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std']...


,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std,...,mfcc_13_std,zcr_mean,zcr_std,rms_mean,rms_std,duration,harmonic_ratio,spectral_centroid_mean,filename,file_path
0,-693.92060,36.645380,201.84492,20.234436,91.13813,12.479002,23.221313,9.360169,-13.558685,6.266790,...,3.122561,0.069015,0.007430,0.001562,0.001273,9.216,0.456694,181.594304,40512331_8.1_1_p1_3544.wav,..\sound_data\sprsound\test\BioCAS2022\test202...
1,-673.22570,23.022497,215.85005,20.453342,82.83959,10.271329,10.603557,12.759145,-17.669603,7.866694,...,2.838013,0.084526,0.011910,0.001940,0.000884,9.216,0.440487,194.822624,40512331_8.1_1_p1_3548.wav,..\sound_data\sprsound\test\BioCAS2022\test202...
2,-574.93840,31.327068,165.27191,29.017046,83.72751,7.765776,14.052708,13.199431,-13.801760,9.057694,...,3.834950,0.079964,0.012061,0.004532,0.008638,9.216,0.337545,197.863467,40512331_8.1_1_p1_3552.wav,..\sound_data\sprsound\test\BioCAS2022\test202...
3,-582.47880,50.437298,177.75015,38.500492,82.81769,11.021469,11.607190,17.249561,-13.789204,8.344293,...,2.739950,0.072714,0.011894,0.005014,0.008968,9.216,0.358471,194.540036,40512331_8.1_1_p2_3545.wav,..\sound_data\sprsound\test\BioCAS2022\test202...
4,-657.68713,51.060295,223.81671,26.085247,85.32820,17.780127,7.967189,15.699356,-18.532220,10.713668,...,3.415674,0.081804,0.012830,0.002690,0.002133,9.216,0.411209,200.951996,40512331_8.1_1_p4_3543.wav,..\sound_data\sprsound\test\BioCAS2022\test202...


## 3. Load Labels from Annotations

In [16]:
# Try to load existing labels
try:
    # If you have a labels file
    sprsound_labels = pd.read_csv("../../sound_data/features/sprsound_labels.csv")
    print("Loaded existing labels")
except:
    print("No labels file found. Creating labels from filename patterns...")
    
    # Create labels based on filename patterns
    # This is a simplified approach - adjust based on your actual labeling
    
    def extract_label_from_filename(filename):
        filename = str(filename).lower()
        if 'wheeze' in filename:
            return 'wheeze'
        elif 'crackle' in filename:
            return 'crackles'
        elif 'rhonchi' in filename:
            return 'rhonchi'
        elif 'stridor' in filename:
            return 'stridor'
        elif 'normal' in filename:
            return 'normal'
        else:
            return 'other'
    
    sprsound_features['symptom'] = sprsound_features['filename'].apply(extract_label_from_filename)
    sprsound_labels = sprsound_features[['filename', 'symptom']].copy()

print("\n Label distribution:")
print(sprsound_labels['symptom'].value_counts())

No labels file found. Creating labels from filename patterns...

 Label distribution:
symptom
other    6567
Name: count, dtype: int64


## 4. Check for Missing Labels

In [17]:
unknown_count = (sprsound_features['symptom'] == 'other').sum()
if unknown_count > 0:
    print(f"\n {unknown_count} files have 'other' label. These are:")
    for f in sprsound_features[sprsound_features['symptom'] == 'other']['filename'].head(10):
        print(f"  - {f}")
    if unknown_count > 10:
        print(f"  ... and {unknown_count-10} more")


 6567 files have 'other' label. These are:
  - 40512331_8.1_1_p1_3544.wav
  - 40512331_8.1_1_p1_3548.wav
  - 40512331_8.1_1_p1_3552.wav
  - 40512331_8.1_1_p2_3545.wav
  - 40512331_8.1_1_p4_3543.wav
  - 40512331_8.1_1_p4_3547.wav
  - 40512331_8.1_1_p4_3551.wav
  - 40638274_9.7_1_p1_1696.wav
  - 40638274_9.7_1_p2_1729.wav
  - 40638274_9.7_1_p2_1775.wav
  ... and 6557 more


## 5. Select Feature Labels

In [18]:
# Select feature columns (exclude metadata columns)
feature_cols = [c for c in sprsound_features.columns if c.startswith(('mfcc', 'zcr', 'rms', 'harmonic', 'spectral', 'duration'))]
print(f"\n Selected {len(feature_cols)} features")
print(f"First 10 features: {feature_cols[:10]}")

# Create feature matrix
X = sprsound_features[feature_cols].values
y = sprsound_features['symptom'].values

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")


 Selected 33 features
First 10 features: ['mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std']

Feature matrix shape: (6567, 33)
Target shape: (6567,)


## 6. Handle Missing Values

In [19]:
# Check for NaN/Inf
print("\n Checking for invalid values:")
print(f"NaN in X: {np.isnan(X).any()}")
print(f"Inf in X: {np.isinf(X).any()}")

# Replace NaN/Inf with 0
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
print("Fixed invalid values")


 Checking for invalid values:
NaN in X: True
Inf in X: False
Fixed invalid values


## 7. Encode Labels

In [20]:
# Encode string labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("\n Class mapping:")
for i, class_name in enumerate(label_encoder.classes_):
    count = np.sum(y_encoded == i)
    percentage = 100 * count / len(y_encoded)
    print(f"  {i}: {class_name:<15} {count:4d} samples ({percentage:5.2f}%)")

# Save encoder
joblib.dump(label_encoder, "../models/sprsound_label_encoder.pkl")
print("\n Label encoder saved")


 Class mapping:
  0: other           6567 samples (100.00%)

 Label encoder saved


## 8. Create Risk Labels

In [21]:
# Define high-risk classes (adjust based on clinical knowledge)
high_risk_classes = ['stridor', 'wheeze']

# Create binary risk labels
y_risk = np.array([1 if label in high_risk_classes else 0 for label in y])

print("\nRisk label distribution:")
print(f"Low risk (0): {np.sum(y_risk == 0)} samples ({100 * np.sum(y_risk == 0)/len(y_risk):.1f}%)")
print(f"High risk (1): {np.sum(y_risk == 1)} samples ({100 * np.sum(y_risk == 1)/len(y_risk):.1f}%)")


Risk label distribution:
Low risk (0): 6567 samples (100.0%)
High risk (1): 0 samples (0.0%)


## 9. Train/Val/Test Split


In [27]:
# First split: train + temp
X_train, X_temp, y_train, y_temp, y_risk_train, y_risk_temp = train_test_split(
    X, y_encoded, y_risk, test_size=0.3, random_state=42, stratify=y_encoded
)

# Second split: validation + test
X_val, X_test, y_val, y_test, y_risk_val, y_risk_test = train_test_split(
    X_temp, y_temp, y_risk_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("\n" + "="*60)
print("DATA SPLIT")
print("="*60)
print(f"Train set: {len(X_train)} samples ({100*len(X_train)/len(X):.1f}%)")
print(f"Val set:   {len(X_val)} samples ({100*len(X_val)/len(X):.1f}%)")
print(f"Test set:  {len(X_test)} samples ({100*len(X_test)/len(X):.1f}%)")


DATA SPLIT
Train set: 4596 samples (70.0%)
Val set:   985 samples (15.0%)
Test set:  986 samples (15.0%)


## 10. Feature Scaling

In [28]:
# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled")
print(f"X_train mean after scaling: {X_train_scaled.mean():.2f}")
print(f"X_train std after scaling:  {X_train_scaled.std():.2f}")

# Save scaler
joblib.dump(scaler, "../models/sprsound_scaler.pkl")
print("Scaler saved")


Features scaled
X_train mean after scaling: -0.00
X_train std after scaling:  1.00
Scaler saved


## 11. Class Weights

In [29]:
# Compute class weights for loss function
class_weights = compute_class_weight(
    'balanced', 
    classes=np.unique(y_train), 
    y=y_train
)

print("\nClass weights for loss function:")
for i, weight in enumerate(class_weights):
    print(f"  {label_encoder.classes_[i]}: {weight:.4f}")

# Compute sample weights for training
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print(f"\nSample weights stats:")
print(f"  Min: {sample_weights.min():.2f}")
print(f"  Max: {sample_weights.max():.2f}")
print(f"  Mean: {sample_weights.mean():.2f}")


Class weights for loss function:
  other: 1.0000

Sample weights stats:
  Min: 1.00
  Max: 1.00
  Mean: 1.00


## 12. Save

In [30]:
# Save scaled features
np.save("../../sound_data/model_data/X_train.npy", X_train_scaled)
np.save("../../sound_data/model_data/X_val.npy", X_val_scaled)
np.save("../../sound_data/model_data/X_test.npy", X_test_scaled)

# Save labels
np.save("../../sound_data/model_data/y_train.npy", y_train)
np.save("../../sound_data/model_data/y_val.npy", y_val)
np.save("../../sound_data/model_data/y_test.npy", y_test)

# Save risk labels
np.save("../../sound_data/model_data/y_risk_train.npy", y_risk_train)
np.save("../../sound_data/model_data/y_risk_val.npy", y_risk_val)
np.save("../../sound_data/model_data/y_risk_test.npy", y_risk_test)

# Save sample weights
np.save("../../sound_data/model_data/sample_weights.npy", sample_weights)

# Save metadata
metadata = {
    'feature_names': feature_cols,
    'class_names': label_encoder.classes_.tolist(),
    'n_classes': len(label_encoder.classes_),
    'n_features': len(feature_cols),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'class_weights': class_weights.tolist(),
    'high_risk_classes': high_risk_classes
}

import json
with open('../../sound_data/model_data/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\n" + "="*60)
print("ALL DATA SAVED SUCCESSFULLY")
print("="*60)
print(f"\nFiles saved to: ../../sound_data/model_data/")


ALL DATA SAVED SUCCESSFULLY

Files saved to: ../../sound_data/model_data/


## 13. Quick Verification

In [31]:
# Load back and verify
X_test_loaded = np.load("../../sound_data/model_data/X_test.npy")
y_test_loaded = np.load("../../sound_data/model_data/y_test.npy")

print("\nVerification:")
print(f"X_test shape: {X_test_loaded.shape} (expected {X_test_scaled.shape})")
print(f"y_test shape: {y_test_loaded.shape} (expected {y_test.shape})")
print(f"Class distribution in test set:")
unique, counts = np.unique(y_test_loaded, return_counts=True)
for i, count in zip(unique, counts):
    print(f"  Class {i}: {count} samples")

print("\nAll good! Ready to train models.")


Verification:
X_test shape: (986, 33) (expected (986, 33))
y_test shape: (986,) (expected (986,))
Class distribution in test set:
  Class 0: 986 samples

All good! Ready to train models.
